# Lab 16 — Distribution Shift & Generalization

*AI Cybersecurity Masterclass · Lab 16 · AI60secLab*

**Companion video:** [▶ Why Security AI Gets Dumber Over Time](https://www.youtube.com/watch?v=7BqRTjWFUBw&list=PLG4l6Ssi9A00)

**Research question:** How different are the prepared UNSW-NB15 training and test distributions, and can a classifier distinguish which dataset a row came from?

A domain-classifier AUC near 0.5 indicates the sampled domains are difficult to distinguish. Higher AUC means stronger detectable differences.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.drift import (
    normalized_median_shift,
    domain_classifier_auc,
)

train, test = load_unsw(DATA_DIR)

X_train, _ = split_xy(train, "label")
X_test, _ = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
numeric_columns = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

marginal_shift = normalized_median_shift(
    X_train,
    X_test,
    numeric_columns,
)

display(marginal_shift.head(20).round(4))

In [ ]:
domain_auc, domain_model = domain_classifier_auc(
    X_train,
    X_test,
    sample_per_domain=20000,
    random_state=42,
)

print(
    "Domain-classifier validation ROC-AUC:",
    round(domain_auc, 4),
)

In [ ]:
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame

y_train = train["label"]
y_test = test["label"]

security_model = random_forest_pipeline(X_train)
security_model.fit(X_train, y_train)

pred_train = security_model.predict(X_train)
score_train = security_model.predict_proba(X_train)[:, 1]

pred_test = security_model.predict(X_test)
score_test = security_model.predict_proba(X_test)[:, 1]

comparison = pd.concat([
    binary_metrics_frame(
        "Training distribution",
        binary_metrics(y_train, pred_train, score_train),
    ),
    binary_metrics_frame(
        "Held-out test distribution",
        binary_metrics(y_test, pred_test, score_test),
    ),
])

display(comparison.round(4))

## Interpretation

Which features shifted most? How distinguishable are the train and test domains? Did security-model performance change across them?

**Takeaway:** unfamiliar data can appear without an adversary; generalization must be measured separately from clean training performance.